# CHUYÊN ĐỀ ĐỈNH CAO: DYNAMIC PARTITION PRUNING (DPP) TRÊN MICROSOFT FABRIC
## CẮT TỈA PHÂN VÙNG ĐỘNG LÚC RUNTIME – VŨ KHÍ TĂNG TỐC QUERY 5 - 15 LẦN

### 1. Bối cảnh & Vấn đề lớn trong thực tế:
- Bạn có bảng giao dịch khổng lồ: **`fact_orders`** (chứa hàng chục triệu đơn hàng, hàng Terabytes dữ liệu), được phân vùng theo ngày: `PARTITIONED BY (order_date)` $\rightarrow$ gồm nhiều thư mục phân vùng ngày.
- Bạn có bảng lịch nhỏ: **`dim_date`** (chứa thông tin ngày, thứ, ngày lễ `is_holiday`).
- Bạn chạy câu truy vấn kinh điển:
  ```sql
  SELECT f.order_id, f.amount, d.holiday_name
  FROM fact_orders f
  JOIN dim_date d ON f.order_date = d.calendar_date
  WHERE d.is_holiday = TRUE;
  ```

### 2. Sự khác biệt cốt lõi giữa Cách Chạy Cũ và DPP (Từ Spark 3.0+):
- **Cách chạy cũ (Khi Tắt DPP):**
  Điều kiện `WHERE is_holiday = TRUE` nằm ở bảng `dim_date`, chứ **KHÔNG NẰM Ở BẢNG FACT**. Spark không thể biết trước ngày nào là ngày lễ khi lập kế hoạch quét bảng Fact $\rightarrow$ Spark bắt buộc phải **quét toàn bộ 100% các thư mục phân vùng ngày**, đưa hết vào bộ nhớ rồi mới Join và loại bỏ các ngày không phải ngày lễ! Cực kỳ lãng phí I/O đĩa và CPU!
- **Cơ chế Dynamic Partition Pruning (DPP):**
  1. Spark quét bảng nhỏ `dim_date` trước, lọc ra các ngày có `is_holiday = TRUE` (ví dụ chỉ có 2 ngày lễ).
  2. Spark đóng gói danh sách 2 ngày này gửi xuống bộ đọc file (FileScan) của bảng Fact ngay lúc Runtime.
  3. Bảng `fact_orders` **cắt tỉa (prune) và bỏ qua ngay lập tức 90% - 98% các thư mục ngày khác**, chỉ mở đúng 2 thư mục ngày lễ để đọc!
  $\rightarrow$ **Giảm 90%+ I/O đĩa, tốc độ tăng gấp nhiều lần!**

> **LƯU Ý CỰC KỲ QUAN TRỌNG VỀ QUY MÔ DỮ LIỆU (BENCHMARK PRINCIPLE):**
> Các kỹ thuật tối ưu hóa phân tán như DPP **chỉ phát huy sức mạnh vượt trội khi dữ liệu đủ lớn** (từ hàng triệu dòng trở lên). Với tập dữ liệu quá nhỏ (vài chục nghìn dòng ~ vài MB), chi phí điều phối mạng và broadcast subquery sẽ lấn át thời gian đọc đĩa!

## BƯỚC 1: KHỞI TẠO 10 TRIỆU DÒNG DỮ LIỆU (30 NGÀY PHÂN VÙNG) TRÊN `lh_silver.dbo`
- **Bảng Fact:** `dpp_fact_orders_large` (10,000,000 đơn hàng rải đều trên 30 ngày: từ `2026-01-01` đến `2026-01-30`).
- **Bảng Dim:** `dpp_dim_date_large` (30 ngày trong tháng 1/2026, chỉ có **2 ngày lễ**: ngày 01/01 và 02/01).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

LH_SILVER = "lh_silver.dbo"
FACT_TABLE = f"{LH_SILVER}.dpp_fact_orders_large"
DIM_TABLE  = f"{LH_SILVER}.dpp_dim_date_large"

# 1. Khởi tạo Bảng Dim Date (30 ngày, chỉ có 2 ngày lễ)
dates_data = [
    (f"2026-01-{i:02d}", True if i in [1, 2] else False, "Tết Dương Lịch" if i in [1, 2] else "Ngày thường")
    for i in range(1, 31)
]
dim_schema = StructType([
    StructField("calendar_date", StringType(), False),
    StructField("is_holiday", BooleanType(), False),
    StructField("holiday_name", StringType(), True)
])
spark.createDataFrame(dates_data, dim_schema).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_TABLE)

# 2. Sinh 10,000,000 dòng dữ liệu bảng Fact rải đều trên 30 ngày (~333,000 đơn/ngày)
print("=== ĐANG SINH 10,000,000 DÒNG DỮ LIỆU BẢNG FACT (30 PHÂN VÙNG NGÀY) ===")
t0 = time.time()

df_fact_source = spark.range(0, 10000000) \
    .withColumn("day_num", F.format_string("%02d", (F.col("id") % 30 + 1))) \
    .withColumn("order_date", F.concat(F.lit("2026-01-"), F.col("day_num"))) \
    .withColumn("order_id", F.concat(F.lit("ORD_"), F.col("id"))) \
    .withColumn("amount", F.round(F.rand() * 1000, 2)) \
    .drop("day_num")

# Ghi ra bảng Delta với phân vùng PARTITIONED BY (order_date)
df_fact_source.write.format("delta").mode("overwrite") \
    .partitionBy("order_date") \
    .option("overwriteSchema", "true") \
    .saveAsTable(FACT_TABLE)

print(f"-> Tạo xong 10,000,000 dòng Fact trong: {time.time() - t0:.2f} giây.")


## BƯỚC 2: XÁC THỰC CẤU TRÚC PHÂN VÙNG VÀ KÍCH THƯỚC BẢNG TRÊN FABRIC
Sử dụng lệnh `DESCRIBE DETAIL` để kiểm tra chắc chắn:
1. `partitionColumns` có chứa `[order_date]` không.
2. Số lượng file (`numFiles`) và kích thước bảng (`sizeInBytes`).

In [ ]:
# Kiểm tra metadata chi tiết của bảng Fact
spark.sql(f"DESCRIBE DETAIL {FACT_TABLE}") \
    .select("format", "partitionColumns", "numFiles", "sizeInBytes") \
    .show(truncate=False)

print("Danh sách ngày lễ cần lọc trong dim_date:")
spark.table(DIM_TABLE).filter("is_holiday = TRUE").show(truncate=False)


## BƯỚC 3: THỬ NGHIỆM KHI TẮT DYNAMIC PARTITION PRUNING (CÁCH CHẠY CŨ)

Ta tắt cờ DPP bằng cấu hình:
`spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "false")`

👉 **Hành vi thực tế:**
Spark không thể biết trước ngày nào là ngày lễ, buộc phải **quét sạch toàn bộ 30 thư mục phân vùng (10 triệu dòng)**, nạp hết lên RAM rồi mới Join để loại bỏ 28 ngày thường!

In [ ]:
# TẮT TÍNH NĂNG DPP
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "false")

query_orders = f"""
    SELECT f.order_id, f.amount, f.order_date, d.holiday_name
    FROM {FACT_TABLE} f
    JOIN {DIM_TABLE} d ON f.order_date = d.calendar_date
    WHERE d.is_holiday = TRUE
"""

df_no_dpp = spark.sql(query_orders)

# Đo thời gian chạy
t0 = time.time()
count_no_dpp = df_no_dpp.count()
time_no_dpp = time.time() - t0

print("=== [KẾT QUẢ KHI TẮT DPP - FULL SCAN 30 PHÂN VÙNG] ===")
print(f"- Số dòng đơn hàng ngày lễ tìm thấy: {count_no_dpp:,} dòng")
print(f"- Thời gian thực thi: {time_no_dpp:.2f} giây (Phải quét toàn bộ 10,000,000 dòng trên 30 thư mục ngày)")


## BƯỚC 4: THỬ NGHIỆM KHI BẬT DYNAMIC PARTITION PRUNING (CẮT TỈA ĐỘNG TỐI ƯU)

Ta kích hoạt các cấu hình tối ưu DPP của Spark và Delta Lake:
- `spark.sql.optimizer.dynamicPartitionPruning.enabled = true`
- `spark.sql.optimizer.dynamicPartitionPruning.useStats = false` (bỏ qua kiểm tra thống kê khắt khe)
- `spark.databricks.optimizer.deltaTableFilesThreshold = 0` (hạ ngưỡng chặn số file của Delta Lake)
- `spark.databricks.optimizer.deltaTableSizeThreshold = 0` (hạ ngưỡng chặn dung lượng 10GB của Delta Lake)

👉 **Phép màu xảy ra lúc Runtime:**
1. Spark quét bảng nhỏ `dim_date` lấy ra đúng 2 ngày lễ (`2026-01-01` và `2026-01-02`).
2. Bắn 2 ngày này vào bộ đọc FileScan của `dpp_fact_orders_large`.
3. Bảng Fact lập tức **BỎ QUA 28 THƯ MỤC PHÂN VÙNG NGÀY THƯỜNG**, chỉ mở đúng 2 thư mục ngày lễ để đọc (~666,000 dòng)!
$\rightarrow$ **Cắt giảm hơn 93% khối lượng dữ liệu I/O cần đọc!**

In [ ]:
# BẬT TÍNH NĂNG DPP VÀ CẤU HÌNH TỐI ƯU DELTA LAKE TRÊN FABRIC
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.useStats", "false")
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.fallbackFilterRatio", "1.0")
spark.conf.set("spark.databricks.optimizer.deltaTableFilesThreshold", "0")
spark.conf.set("spark.databricks.optimizer.deltaTableSizeThreshold", "0")

df_with_dpp = spark.sql(query_orders)

# Đo thời gian chạy
t0 = time.time()
count_with_dpp = df_with_dpp.count()
time_with_dpp = time.time() - t0

print("=== [KẾT QUẢ KHI BẬT DPP - CHỈ QUÉT ĐÚNG 2 PHÂN VÙNG NGÀY LỄ] ===")
print(f"- Số dòng đơn hàng ngày lễ tìm thấy: {count_with_dpp:,} dòng")
print(f"- Thời gian thực thi: {time_with_dpp:.2f} giây (Chỉ đọc ~666,000 dòng của 2 ngày lễ, bỏ qua 28 ngày còn lại)")
print(f"\n=> TỐC ĐỘ CẢI THIỆN VƯỢT TRỘI: DPP NHANH HƠN {time_no_dpp / time_with_dpp:.2f} LẦN!")

# In Physical Plan thực tế sau khi action hoàn tất (Lúc này AQE đã hoàn thành kế hoạch Runtime)
print("\n=== PHYSICAL PLAN THỰC THI THỰC TẾ ===")
df_with_dpp.explain("formatted")


## BƯỚC 5: TỔNG KẾT & CHECKLIST 4 ĐIỀU KIỆN ĐỂ DPP TỰ ĐỘNG KÍCH HOẠT

### 1. Bảng So Sánh Hiệu Năng Thực Tế:
| Tiêu Chí | Khi TẮT DPP | Khi BẬT DPP | Mức Độ Tối Ưu |
| :--- | :--- | :--- | :--- |
| **Số phân vùng ngày phải quét** | 30 / 30 folders (100%) | **2 / 30 folders (~6.7%)** | **Bỏ qua 93.3% thư mục** |
| **Số dòng dữ liệu nạp vào RAM** | 10,000,000 dòng | **~666,666 dòng** | **Giảm 93.3% RAM & I/O** |
| **Thời gian thực thi** | Chậm (quét toàn bộ đĩa) | **Nhanh hơn gấp nhiều lần** | **Tiết kiệm Fabric Capacity Unit (CU)** |

### 2. Checklist 4 Điều Kiện Bắt Buộc Của Senior Data Engineer:
| STT | Điều Kiện Bắt Buộc | Giải Thích & Gotchas Thực Tế |
| :---: | :--- | :--- |
| **1** | **Bảng Fact bắt buộc phải PARTITIONED BY** | Cột join phải nằm trong mệnh đề phân vùng của bảng Fact. Nếu bảng Fact là bảng phẳng không phân vùng $\rightarrow$ DPP vô hiệu. |
| **2** | **Cột Join phải đúng là Cột Phân Vùng** | Điều kiện `ON fact.order_date = dim.calendar_date` phải dùng đúng cột partition. Nếu Join theo `customer_id` thì không thể tỉa theo ngày. |
| **3** | **Loại phép Join phải hợp lệ** | DPP chỉ hoạt động với `INNER JOIN`, `LEFT SEMI JOIN` (hoặc `RIGHT OUTER JOIN` nếu Fact nằm bên phải). Phép `FULL OUTER JOIN` không thể cắt tỉa vì phải giữ lại dữ liệu cả 2 bên. |
| **4** | **Bảng Dimension phải có Filter chọn lọc** | Mệnh đề `WHERE` trên bảng Dim phải lọc bớt một lượng ngày đáng kể (`WHERE is_holiday = TRUE`). Nếu `WHERE` lấy toàn bộ 30 ngày thì không có gì để tỉa! |